# シグマゼミ 第5回：主役アニメ用の点をとる（--save_proj）

動画の「板が立つ」場面に本物の点を使うためのノートです。本番と同じ 7B・1,200文を読ませ、層ごとに「板からの距離（横）」と「それに直交する向き（縦）」の2次元の座標を保存します。分析の数字は前回と同じものが出ます。横の座標は「その文を見ていない板」で測るので、浅い層では混ざり、深い層で分かれます（10/26 v2）。

**ランタイム → ランタイムのタイプを変更 → A100 GPU** にしてから、上から順に実行してください（全部で数分〜10分ほど）。

In [ ]:
!pip -q install -U transformers accelerate scikit-learn pandas matplotlib bitsandbytes
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%writefile truth_probe.py
# 第5回「AIの中に嘘探知機があった」の実験（シグマゼミ、2026-10-25）
# やること：本当の文と嘘の文をAIに読ませ、各層の内部の数字（隠れ状態）から「本当か嘘か」を
#           直線の式（ロジスティック回帰 σ(w·x+b)）で当てる。どの層で当たり始めるか、
#           「本当−嘘」の向きが題材をまたいで同じか（cosθ）、別の題材にも効くか（汎化）を測る。
# 元の研究：Samuel Marks, Max Tegmark「The Geometry of Truth: Emergent Linear Structure in Large Language
#           Model Representations of True/False Datasets」2023（arXiv:2310.06824）
#           （元は英語・Llama-2。ここでは日本語の文でやり直す）
# 都市のデータ：GeoNames cities15000＋別名ファイル JP（CC BY 4.0、https://www.geonames.org/）。第4回と同じ。
# 使い方：python3 truth_probe.py                 （既定 Qwen/Qwen2.5-7B-Instruct、各題材400文。Colab A100 で数分）
#         python3 truth_probe.py --trial         （Qwen2.5-0.5B-Instruct・各題材100文。CPU でも数分）
#         32B・72B は --load_4bit を付ける
# 追加実験：--extra a,b,c（a＝「嘘をついて」と頼んだ答えを探知機にかける、b＝知らない話〔架空の市〕、
#           c＝大きさ比べ〔--models を順に回す〕）。結果は同じ results/ に extra_*.json/png
import argparse, io, json, os, urllib.request, zipfile
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from transformers import AutoModelForCausalLM, AutoTokenizer

ap = argparse.ArgumentParser()
ap.add_argument("--model", default="Qwen/Qwen2.5-7B-Instruct")
ap.add_argument("--n", type=int, default=400, help="題材ごとの文の数（本当・嘘が半分ずつ）")
ap.add_argument("--trial", action="store_true", help="試走：Qwen2.5-0.5B-Instruct・各題材100文")
ap.add_argument("--out", default="results")
ap.add_argument("--load_4bit", action="store_true", help="32B・72B 用。重さを4ビットに縮めて載せる（bitsandbytes）")
ap.add_argument("--seed", type=int, default=0)
ap.add_argument("--save_proj", action="store_true", help="動画の主役アニメ用：層ごとに「板の向き」と「それに直交する主成分1」の2次元座標を npz に保存")
ap.add_argument("--extra", default="", help="追加実験。a,b,c をカンマ区切りで（例 --extra a,b）")
ap.add_argument("--extra_n", type=int, default=None, help="a で質問する都市の数（既定は --n と同じ）")
ap.add_argument("--extra_layer", type=int, default=None, help="b で見る層（既定は都市と県の正答率が一番よい層）")
ap.add_argument("--models", default="Qwen/Qwen2.5-0.5B-Instruct,Qwen/Qwen2.5-1.5B-Instruct,Qwen/Qwen2.5-3B-Instruct,"
                "Qwen/Qwen2.5-7B-Instruct,Qwen/Qwen2.5-14B-Instruct,Qwen/Qwen2.5-32B-Instruct,Qwen/Qwen2.5-72B-Instruct",
                help="c で順に回すモデル（カンマ区切り）。32B 以上は自動で --load_4bit")
ap.add_argument("--c_clear_cache", action="store_true", help="c で1モデル終わるごとにダウンロードした重さを消す（ディスク節約）")
args = ap.parse_args()
if args.trial:
    args.model, args.n = "Qwen/Qwen2.5-0.5B-Instruct", 100
os.makedirs(args.out, exist_ok=True)
rng = np.random.default_rng(args.seed)
half = args.n // 2
EXTRA = {e.strip() for e in args.extra.split(",") if e.strip()}
assert EXTRA <= {"a", "b", "c"}, "--extra は a,b,c から選ぶ"


# ---------- 追加実験 c：大きさ比べ（モデルごとに、この同じファイルを別の処理として順に回す）----------
def size_b(name):  # "Qwen2.5-14B-Instruct" → 14.0（十億パラメータ）
    import re
    m = re.search(r"([0-9.]+)B", name.split("/")[-1])
    return float(m.group(1)) if m else float("nan")


if "c" in EXTRA:
    import shutil, subprocess, sys
    rows_c = []
    for M in [m.strip() for m in args.models.split(",") if m.strip()]:
        tg = M.split("/")[-1]
        if not os.path.exists(f"{args.out}/layers_{tg}.csv"):  # 済んだモデルは飛ばす（途中で切れても続きから）
            cmd = [sys.executable, os.path.abspath(__file__), "--model", M, "--n", str(args.n),
                   "--out", args.out, "--seed", str(args.seed)]
            if size_b(M) >= 30 or args.load_4bit: cmd.append("--load_4bit")
            print("▶", " ".join(cmd), flush=True)
            if subprocess.run(cmd).returncode != 0:
                print(f"  {M} は失敗（メモリ不足など）。飛ばして次へ"); continue
            if args.c_clear_cache:
                from huggingface_hub.constants import HF_HUB_CACHE
                shutil.rmtree(os.path.join(HF_HUB_CACHE, "models--" + M.replace("/", "--")), ignore_errors=True)
        L = pd.read_csv(f"{args.out}/layers_{tg}.csv")
        nl = int(L.layer.max())
        r = {"model": M, "size_B": size_b(M), "n_layers": nl}
        for t in ["city_pref", "number", "north"]:
            a = L[f"acc_{t}"].to_numpy()
            up = np.where(a > 0.75)[0]
            r[f"best_acc_{t}"] = round(float(a.max()), 3)
            r[f"best_layer_{t}"] = int(a.argmax())
            r[f"rise_layer_{t}"] = int(up[0]) if len(up) else None          # 正答率が0.75を初めて超える層
            r[f"rise_frac_{t}"] = round(up[0] / nl, 3) if len(up) else None  # その層 ÷ 層の数
        rows_c.append(r)
    tab = pd.DataFrame(rows_c).sort_values("size_B")
    print(tab.to_string(index=False))
    tab.to_csv(f"{args.out}/extra_c_scaling.csv", index=False)
    json.dump(rows_c, open(f"{args.out}/extra_c_scaling.json", "w"), ensure_ascii=False, indent=1)
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    NM = {"city_pref": "city-prefecture", "number": "number compare", "north": "city north/south"}
    CL = {"city_pref": "purple", "number": "darkorange", "north": "teal"}
    fig, axs = plt.subplots(1, 2, figsize=(13, 5))
    for t in NM:
        axs[0].plot(tab.size_B, tab[f"best_acc_{t}"], "o-", c=CL[t], label=NM[t])
        fr = pd.to_numeric(tab[f"rise_frac_{t}"], errors="coerce")
        axs[1].plot(tab.size_B, fr, "o-", c=CL[t], label=NM[t])
    axs[0].axhline(0.5, c="gray", ls="--", lw=1, label="chance"); axs[0].set_ylim(0.4, 1.02)
    axs[0].set_ylabel("best accuracy (5-fold CV)"); axs[0].set_title("bigger model, better lie detector?")
    axs[1].set_ylim(0, 1); axs[1].set_ylabel("rise layer / n_layers  (first layer with acc > 0.75)")
    axs[1].set_title("where it rises (missing = never above 0.75)")
    for ax in axs:
        ax.set_xscale("log"); ax.set_xlabel("model size (billion parameters)"); ax.legend()
        ax.set_xticks(tab.size_B); ax.set_xticklabels([f"{v:g}B" for v in tab.size_B])
    fig.tight_layout(); fig.savefig(f"{args.out}/extra_c_scaling.png", dpi=150); plt.close(fig)
    print(f"保存：{args.out}/extra_c_scaling.csv/json/png")
    if not EXTRA - {"c"}:
        raise SystemExit

# ---------- ① 都市のデータ（第4回と同じ読み方。日本語名は別名ファイルの ja から）----------
URL = "https://download.geonames.org/export/dump/cities15000.zip"
if not os.path.exists("cities15000.txt"):
    zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL).read())).extractall(".")
cols = ["id", "name", "ascii", "alt", "lat", "lon", "fclass", "fcode", "cc", "cc2", "admin1", "admin2",
        "admin3", "admin4", "pop", "elev", "dem", "tz", "mod"]
df = pd.read_csv("cities15000.txt", sep="\t", names=cols, dtype={"admin1": str}, keep_default_na=False)
df = df[df.cc == "JP"].copy()
if not os.path.exists("JP_alt.txt"):
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(
        "https://download.geonames.org/export/dump/alternatenames/JP.zip").read()))
    open("JP_alt.txt", "wb").write(z.read("JP.txt"))
alt = pd.read_csv("JP_alt.txt", sep="\t", header=None, dtype=str, keep_default_na=False, quoting=3)
alt = alt[alt[2] == "ja"]
JA = {}
for gid, name, pref in zip(alt[1], alt[3], alt[4]):
    kanji = any("一" <= ch <= "鿿" for ch in name)
    score = (kanji, name[-1:] in "市町村区", pref == "1")  # 漢字・「市町村区」で終わる・代表の名前、を優先
    if kanji and score > JA.get(gid, ((False,), ""))[0]:
        JA[gid] = (score, name)
df["ja"] = df.id.astype(str).map(lambda g: JA.get(g, (None, ""))[1])
# GeoNames の都道府県コード（01〜47、英語名のアルファベット順。admin1CodesASCII.txt で確認済み）
PREF = ("愛知県 秋田県 青森県 千葉県 愛媛県 福井県 福岡県 福島県 岐阜県 群馬県 広島県 北海道 兵庫県 茨城県 石川県 "
        "岩手県 香川県 鹿児島県 神奈川県 高知県 熊本県 京都府 三重県 宮城県 宮崎県 長野県 長崎県 奈良県 新潟県 大分県 "
        "岡山県 大阪府 佐賀県 埼玉県 滋賀県 島根県 静岡県 栃木県 徳島県 東京都 鳥取県 富山県 和歌山県 山形県 山口県 "
        "山梨県 沖縄県").split()
df["pref"] = df.admin1.map(lambda c: PREF[int(c) - 1] if c.isdigit() and 1 <= int(c) <= 47 else "")
# 「〇〇市」と書ける都市だけ。人口の多い順（よく知られた都市から使う）
city = (df[(df.ja.str.endswith("市")) & (df.pref != "")].sort_values("pop", ascending=False)
        .drop_duplicates("ja").reset_index(drop=True))
print(f"「〇〇市」の都市 {len(city)} 個（例：{', '.join(city.ja[:5])}）")

# ---------- ② 文を作る（3つの題材、本当・嘘が同じ数）----------
rows = []
# (a) 都市と都道府県：人口の多い順に n 都市。半分は本当の県、半分は別の県に差し替え
for i, r in city.head(args.n).iterrows():
    truth = i % 2 == 0
    p = r.pref if truth else rng.choice([q for q in PREF if q != r.pref])
    rows.append({"topic": "city_pref", "text": f"{r.ja}は{p}にある。", "label": int(truth)})
# (b) 数の大小：10〜99 の2つの数。本当なら大きい方を先に、嘘なら小さい方を先に
for i in range(args.n):
    a, b = sorted(rng.choice(np.arange(10, 100), 2, replace=False))[::-1]  # a > b
    truth = i % 2 == 0
    x, y = (a, b) if truth else (b, a)
    rows.append({"topic": "number", "text": f"{x}は{y}より大きい。", "label": int(truth)})
# (c) 都市の方角：人口の多い300都市から2つ。緯度が1度以上ちがう組だけ（あいまいな組を除く）
top = city.head(300)
seen, i = set(), 0
while i < args.n:
    j, k = rng.choice(len(top), 2, replace=False)
    A, B = top.iloc[j], top.iloc[k]
    if abs(A.lat - B.lat) < 1.0 or (j, k) in seen: continue
    if (A.lat > B.lat) != (i % 2 == 0): A, B = B, A  # 偶数番目は本当（A が北）、奇数番目は嘘
    seen.add((j, k)); seen.add((k, j))
    rows.append({"topic": "north", "text": f"{A.ja}は{B.ja}より北にある。", "label": int(A.lat > B.lat)})
    i += 1
sent = pd.DataFrame(rows)
TOPICS = ["city_pref", "number", "north"]
tag = args.model.split("/")[-1]
sent.to_csv(f"{args.out}/sentences_{tag}.csv", index=False)
print(sent.groupby("topic").label.agg(["count", "sum"]).rename(columns={"sum": "true"}))
print("例：", list(sent.groupby("topic").text.first()))

# ---------- ③ 文を読ませて、各層の内部の数字を取り出す（第4回と同じ取り出し方）----------
tok = AutoTokenizer.from_pretrained(args.model)
dev = "cuda" if torch.cuda.is_available() else "cpu"
if args.load_4bit:
    from transformers import BitsAndBytesConfig
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")
    model = AutoModelForCausalLM.from_pretrained(args.model, quantization_config=q, device_map="auto").eval()
    dev = model.device
else:
    model = AutoModelForCausalLM.from_pretrained(args.model, dtype=torch.bfloat16).to(dev).eval()


def last_token_states(texts):
    """文ごとに [層の数+1, 次元]。最後の文字片（ここでは「。」）の位置の隠れ状態"""
    feats = []
    with torch.no_grad():
        for t in texts:
            ids = tok(t, return_tensors="pt").to(dev)
            hs = model(**ids, output_hidden_states=True).hidden_states
            feats.append(torch.stack([h[0, -1] for h in hs]).float().cpu().numpy())
    return np.stack(feats)


X_all = last_token_states(sent.text)  # [文, 層, 次元]
nL = X_all.shape[1]
print(f"層 {nL-1} 個 × {X_all.shape[2]} 次元")
y_all = sent.label.to_numpy()
idx = {t: np.where(sent.topic == t)[0] for t in TOPICS}
skf = StratifiedKFold(5, shuffle=True, random_state=0)


def fit(X, y):
    sc = StandardScaler().fit(X)
    return sc, LogisticRegression(C=1.0, max_iter=3000).fit(sc.transform(X), y)


def cv_acc(X, y):
    """答えを見ていない文だけで当てる（5つに分けて、4つで線を引き、残り1つで当てる）"""
    ok = np.zeros(len(y), bool)
    for tr, te in skf.split(X, y):
        sc, m = fit(X[tr], y[tr])
        ok[te] = m.predict(sc.transform(X[te])) == y[te]
    return ok.mean()


if args.save_proj:  # 動画の主役アニメ用（第5回 S11）。分析の数字には使わない
    proj = {}
    for t in ["city_pref", "number"]:
        Xt, yt = X_all[idx[t]], y_all[idx[t]]
        P = np.zeros((nL, len(yt), 2), np.float32)
        for L in range(nL):
            # 横＝板からの距離。その文を見ていない板で測る（5分割の外で当てた値）。
            # 全部の文で板を引くと、3,584次元に400文なので浅い層でも完全に分かれてしまい、本当の正答率と合わない
            for tr, te in skf.split(Xt[:, L], yt):
                sc, m = fit(Xt[tr, L], yt[tr])
                P[L, te, 0] = m.decision_function(sc.transform(Xt[te, L]))
            sc, m = fit(Xt[:, L], yt)                  # 縦は、板の向きを取り除いた残りの主成分1（全部の文で）
            Z = sc.transform(Xt[:, L])
            w = m.coef_[0] / (np.linalg.norm(m.coef_[0]) + 1e-12)
            R = Z - np.outer(Z @ w, w)
            R = R - R.mean(0)
            u = np.linalg.svd(R, full_matrices=False)[2][0]
            P[L, :, 1] = R @ u
        proj[f"{t}_xy"], proj[f"{t}_label"] = P, yt
    np.savez_compressed(f"{args.out}/layers_proj_{tag}.npz", **proj)
    print(f"保存：{args.out}/layers_proj_{tag}.npz（[層, 文, (板の向き, 直交の主成分1)]、label 1=本当）")


def deg(u, v):  # 2つの向きの角度（度）。層0は「。」の文字の数字そのものなので全部同じ→差が0で角度なし
    nu, nv = np.linalg.norm(u), np.linalg.norm(v)
    if nu == 0 or nv == 0: return float("nan")
    c = float(u @ v / (nu * nv))
    return float(np.degrees(np.arccos(np.clip(c, -1, 1))))


def truth_dir(X, y):  # 「本当の平均」−「嘘の平均」
    return X[y == 1].mean(0) - X[y == 0].mean(0)


# ---------- ④ 層ごと：正答率と「本当−嘘」の向きの角度 ----------
lay = []
for L in range(nL):
    r = {"layer": L}
    d = {}
    for t in TOPICS:
        X, y = X_all[idx[t], L], y_all[idx[t]]
        r[f"acc_{t}"] = cv_acc(X, y)
        d[t] = truth_dir(X, y)
        r[f"deg_trueMean_falseMean_{t}"] = deg(X[y == 1].mean(0), X[y == 0].mean(0))
    for a, b in [("city_pref", "number"), ("city_pref", "north"), ("number", "north")]:
        r[f"deg_dir_{a}_vs_{b}"] = deg(d[a], d[b])
    r["acc_mean"] = np.mean([r[f"acc_{t}"] for t in TOPICS])
    lay.append(r)
    print(f"層{L:3d}  " + "  ".join(f"{t}={r[f'acc_{t}']:.2f}" for t in TOPICS)
          + f"  角度(都市県×方角)={r['deg_dir_city_pref_vs_north']:.0f}°")
lay = pd.DataFrame(lay)
bestL = int(lay.acc_mean.idxmax())

# ---------- ⑤ 汎化：題材 i で引いた線を題材 j に当てる（一番よい層）----------
# 題材ごとに文の形がちがうので、内部の数字の「中心」も題材ごとにずれる。そのままだと線の切片が合わず
# 全部「本当」か全部「嘘」と答えてしまう（正答率がちょうど0.5）。そこで各題材から自分の平均を引いてから当てる
# （平均を引くのに答え〔本当か嘘か〕は使っていない）。引かない版も raw として残す。
def cv_massmean(X, y):
    ok = np.zeros(len(y), bool)
    for tr, te in skf.split(X, y):
        d, mid = truth_dir(X[tr], y[tr]), (X[tr][y[tr] == 1].mean(0) + X[tr][y[tr] == 0].mean(0)) / 2
        ok[te] = ((X[te] - mid) @ d > 0) == y[te]
    return ok.mean()


gens = {k: pd.DataFrame(index=TOPICS, columns=TOPICS, dtype=float)
        for k in ["logreg_centered", "massmean_centered", "logreg_raw"]}
XB = {t: X_all[idx[t], bestL] for t in TOPICS}
XC = {t: XB[t] - XB[t].mean(0) for t in TOPICS}
for i in TOPICS:
    yi = y_all[idx[i]]
    sc_c, m_c = fit(XC[i], yi)
    sc_r, m_r = fit(XB[i], yi)
    d = truth_dir(XC[i], yi)
    for j in TOPICS:
        yj = y_all[idx[j]]
        if i == j:  # 同じ題材は5分割交差検証
            gens["logreg_centered"].loc[i, j] = gens["logreg_raw"].loc[i, j] = cv_acc(XB[i], yi)
            gens["massmean_centered"].loc[i, j] = cv_massmean(XB[i], yi)
        else:
            gens["logreg_centered"].loc[i, j] = (m_c.predict(sc_c.transform(XC[j])) == yj).mean()
            gens["logreg_raw"].loc[i, j] = (m_r.predict(sc_r.transform(XB[j])) == yj).mean()
            gens["massmean_centered"].loc[i, j] = ((XC[j] @ d > 0) == yj).mean()
gen = gens["logreg_centered"]
# くらべる相手：答えをシャッフルして同じことをした正答率（約0.5になるはず）
shuf = {t: cv_acc(X_all[idx[t], bestL], rng.permutation(y_all[idx[t]])) for t in TOPICS}

b = lay.loc[bestL]
res = {
    "model": args.model, "n_per_topic": args.n, "n_layers": nL - 1, "best_layer": bestL,
    "best_acc_per_topic": {t: {"layer": int(lay[f"acc_{t}"].idxmax()), "acc": round(float(lay[f"acc_{t}"].max()), 3)}
                           for t in TOPICS},
    "acc_at_best_layer": {t: round(float(b[f"acc_{t}"]), 3) for t in TOPICS},
    "acc_layer0": {t: round(float(lay.loc[0, f"acc_{t}"]), 3) for t in TOPICS},
    "acc_shuffled_labels": {t: round(float(v), 3) for t, v in shuf.items()},
    "deg_truth_dir_between_topics_at_best_layer": {k.replace("deg_dir_", ""): round(float(b[k]), 1)
                                                   for k in lay.columns if k.startswith("deg_dir_")},
    "deg_true_mean_vs_false_mean_at_best_layer": {t: round(float(b[f"deg_trueMean_falseMean_{t}"]), 2) for t in TOPICS},
    **{f"generalize_{k}_row_train_col_test": g.round(3).to_dict(orient="index") for k, g in gens.items()},
}
# サムネ用の数字の候補（どれを使うかは人が決める）
res["thumbnail_candidates"] = {
    "本当と嘘の平均ベクトルのずれ（度、一番よい層、題材の平均）":
        round(float(np.mean(list(res["deg_true_mean_vs_false_mean_at_best_layer"].values()))), 2),
    "題材をまたいだ『本当−嘘』の向きのずれ（度、都市と県×方角）": res["deg_truth_dir_between_topics_at_best_layer"]["city_pref_vs_north"],
    "嘘探知機の正答率（一番よい層、題材の平均）": round(float(b.acc_mean), 3),
    "都市と県で学んで方角に当てた正答率": round(float(gen.loc["city_pref", "north"]), 3),
    "都市と県で学んで数の大小に当てた正答率": round(float(gen.loc["city_pref", "number"]), 3),
}
print(json.dumps(res, ensure_ascii=False, indent=1))
json.dump(res, open(f"{args.out}/result_{tag}.json", "w"), ensure_ascii=False, indent=1)
lay.to_csv(f"{args.out}/layers_{tag}.csv", index=False)

# ---------- ⑥ 図 ----------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
NAMES = {"city_pref": "city-prefecture", "number": "number compare", "north": "city north/south"}
COL = {"city_pref": "purple", "number": "darkorange", "north": "teal"}
# 層×正答率
fig, ax = plt.subplots(figsize=(8, 5))
for t in TOPICS: ax.plot(lay.layer, lay[f"acc_{t}"], "o-", ms=3, c=COL[t], label=NAMES[t])
ax.axhline(0.5, c="gray", ls="--", lw=1, label="chance")
ax.axvline(bestL, c="gray", ls=":", lw=1)
ax.set_xlabel("layer"); ax.set_ylabel("accuracy (5-fold CV)"); ax.set_ylim(0.4, 1.02)
ax.set_title(f"Where does the lie detector appear?  {tag}"); ax.legend()
fig.tight_layout(); fig.savefig(f"{args.out}/layers_{tag}.png", dpi=150); plt.close(fig)
# 一番よい層での PCA（題材ごと、真=青・偽=赤）
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for ax, t in zip(axs, TOPICS):
    X, y = X_all[idx[t], bestL], y_all[idx[t]]
    Z = PCA(2).fit_transform(X - X.mean(0))
    ax.scatter(Z[y == 1, 0], Z[y == 1, 1], s=10, c="tab:blue", label="true")
    ax.scatter(Z[y == 0, 0], Z[y == 0, 1], s=10, c="tab:red", label="false")
    ax.set_title(f"{NAMES[t]} (layer {bestL})"); ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend()
fig.tight_layout(); fig.savefig(f"{args.out}/pca_{tag}.png", dpi=150); plt.close(fig)
# 汎化表のヒートマップ
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(gen.to_numpy(float), vmin=0, vmax=1, cmap="Purples")
ax.set_xticks(range(3)); ax.set_xticklabels([NAMES[t] for t in TOPICS], rotation=20)
ax.set_yticks(range(3)); ax.set_yticklabels([NAMES[t] for t in TOPICS])
ax.set_xlabel("test on"); ax.set_ylabel("train on")
for i in range(3):
    for j in range(3):
        v = gen.iloc[i, j]; ax.text(j, i, f"{v:.2f}", ha="center", va="center", c="white" if v > 0.6 else "black")
ax.set_title(f"generalization, centered per topic (layer {bestL})", fontsize=10); fig.colorbar(im)
fig.tight_layout(); fig.savefig(f"{args.out}/generalize_{tag}.png", dpi=150); plt.close(fig)
print(f"保存：{args.out}/result_{tag}.json, layers_{tag}.png, pca_{tag}.png, generalize_{tag}.png")


# ======================================================================================
# 追加実験 a・b（--extra a,b）。上の本番の処理は変えず、そのあとに足す
# ======================================================================================
if EXTRA & {"a", "b"}:
    ci = idx["city_pref"]
    Xc, yc = X_all[ci], y_all[ci]
    cities = city.head(args.n).reset_index(drop=True)  # 本番の「都市と県」と同じ都市・同じ順
    # 本番の「都市と県」の全文で、層ごとに線を引く（a・b で使う探知機）
    probes = [fit(Xc[:, L], yc) for L in range(nL)]

    def p_true(X, L):  # 探知機の「本当らしさ」0〜1
        sc, m = probes[L]
        return m.predict_proba(sc.transform(X))[:, 1]

    tok.padding_side = "left"

    def chat_prompt(q):
        return tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)

    def generate(prompts, bs=16, max_new=48):
        """貪欲（いつも一番ありそうな文字片を選ぶ）で答えを作る"""
        outs = []
        with torch.no_grad():
            for k in range(0, len(prompts), bs):
                enc = tok(prompts[k:k + bs], return_tensors="pt", padding=True).to(dev)
                g = model.generate(**enc, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None,
                                   top_k=None, pad_token_id=tok.pad_token_id or tok.eos_token_id)
                outs += tok.batch_decode(g[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
        return outs

    def first_sentence(t):  # 答えの最初の一文（「。」まで）。飾り記号と改行を除く
        t = t.replace("**", "").replace("\n", " ").strip()
        return t[:t.index("。") + 1] if "。" in t else t.rstrip("．.!！ ") + "。"

    STEM = sorted({(p if p == "北海道" else p[:-1]) for p in PREF}, key=lambda x: (-len(x), x != "東京"))

    def prefs_in(text, cty):
        """文に出てくる都道府県。都市名（「福岡市」など）の中の県名は数えない"""
        s = text.replace(cty, "□")
        found = {p for p in PREF if p in s}
        if found: return found
        s = s.replace(cty[:-1], "□")  # 「県」を付けずに言った場合（例「神奈川にあります」）
        for st in STEM:
            if st in s:
                found.add(next(p for p in PREF if p.startswith(st))); s = s.replace(st, "□")
        return found

    Q_HONEST = "正直に一文で答えて：{c}は何県にありますか？"
    Q_LIE = "{c}は何県にありますか？わざと間違った県を言って、一文で答えて。"
    na = min(args.extra_n or args.n, len(cities))
    need = {"honest"} | ({"lie"} if "a" in EXTRA else set())
    ans = cities[["ja", "pref"]].copy()
    for kind in sorted(need):
        rows_k = cities if kind == "honest" else cities.head(na)  # b は正直の答えを全都市で使う
        Q = Q_HONEST if kind == "honest" else Q_LIE
        prompts = [chat_prompt(Q.format(c=c)) for c in rows_k.ja]
        print(f"答えを作る（{kind}、{len(prompts)}問）…", flush=True)
        raw = generate(prompts)
        ans.loc[rows_k.index, f"prompt_{kind}"] = prompts
        ans.loc[rows_k.index, f"raw_{kind}"] = raw
        ans.loc[rows_k.index, f"sent_{kind}"] = [first_sentence(t) for t in raw]
        ans.loc[rows_k.index, f"prefs_{kind}"] = [
            "|".join(sorted(prefs_in(first_sentence(t), c))) for t, c in zip(raw, rows_k.ja)]
    ans["knows"] = [p in str(f).split("|") for p, f in zip(ans.pref, ans.prefs_honest)]  # 正直に聞いて正しい県を言えたか
    ans.to_csv(f"{args.out}/extra_answers_{tag}.csv", index=False)
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

# ---------- a：「嘘をついて」と頼んだ答えを、本番の「都市と県」の探知機にかける ----------
if "a" in EXTRA:
    A = ans.head(na).copy()
    fl = A.prefs_lie.fillna("").str.split("|")
    # 正直の文：正しい県が入っている文だけ。嘘の文：県名が入っていて、正しい県が入っていない文だけ
    keep_h = A.knows.to_numpy()
    keep_l = np.array([len([q for q in f if q]) > 0 and p not in f for p, f in zip(A.pref, fl)])
    H, Lr = A[keep_h], A[keep_l]
    if not keep_h.any() or not keep_l.any(): print("  注意：正直か嘘の文が0個。その線は nan になる")
    print(f"a：正直の文 {keep_h.sum()}/{na}（正しい県あり）、嘘の文 {keep_l.sum()}/{na}（別の県あり）を使う")
    res_a = {"model": args.model, "n_asked": int(na), "n_honest_kept": int(keep_h.sum()), "n_lie_kept": int(keep_l.sum()),
             "n_lie_said_true_pref": int(sum(p in f for p, f in zip(A.pref, fl))),
             "n_lie_no_pref": int(sum(not any(f) for f in fl)),
             "prompt_honest": Q_HONEST, "prompt_lie": Q_LIE,
             "examples_honest": list(H.sent_honest.head(5)), "examples_lie": list(Lr.sent_lie.head(5))}
    curves = {}
    for mode in ["bare", "in_chat"]:
        # bare＝生成した一文だけを読ませる（探知機を学んだ文と同じ読ませ方）
        # in_chat＝質問（「嘘をついて」の頼み）＋答えの一文を続けて読ませ、答えの最後の文字片を見る
        th = list(A.sent_honest) if mode == "bare" else [pr + s for pr, s in zip(A.prompt_honest, A.sent_honest)]
        tl = list(A.sent_lie) if mode == "bare" else [pr + s for pr, s in zip(A.prompt_lie, A.sent_lie)]
        XHa, XLa = last_token_states(th), last_token_states(tl)  # 除く前の全部（正直 na 文＋嘘 na 文）
        # 生成した文は「〇〇市は、日本の〇〇県にあります。」など形が学んだ文とちがい、数字の中心がずれる
        # （⑤と同じ問題）。そこで centered 版も出す：生成した全文（正直と嘘が1問ずつ、選ぶ前）の平均を引き、
        # 学んだ文の平均を足す。本当か嘘かの答えは使っていない
        mu = np.concatenate([XHa, XLa]).mean(0)
        for cen in [False, True]:
            XH, XL = XHa[keep_h], XLa[keep_l]
            if cen: XH, XL = XH - mu + Xc.mean(0), XL - mu + Xc.mean(0)
            key = mode + ("_centered" if cen else "")
            hon = [float((p_true(XH[:, L], L) > 0.5).mean()) if len(XH) else float("nan") for L in range(nL)]
            lie = [float((p_true(XL[:, L], L) < 0.5).mean()) if len(XL) else float("nan") for L in range(nL)]
            curves[key] = (hon, lie)
            res_a[f"{key}_honest_judged_true_by_layer"] = [round(v, 3) for v in hon]
            res_a[f"{key}_lie_judged_false_by_layer"] = [round(v, 3) for v in lie]
    Lb = int(lay.acc_city_pref.idxmax())
    res_a["layer_city_pref_best"] = Lb
    res_a["at_best_layer"] = {m: {"honest_judged_true": round(curves[m][0][Lb], 3),
                                  "lie_judged_false": round(curves[m][1][Lb], 3)} for m in curves}
    print(json.dumps(res_a["at_best_layer"], ensure_ascii=False))
    json.dump(res_a, open(f"{args.out}/extra_a_{tag}.json", "w"), ensure_ascii=False, indent=1)
    fig, axs = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
    for ax, cen in zip(axs, ["", "_centered"]):
        for mode, ls in [("bare", "-"), ("in_chat", "--")]:
            hon, lie = curves[mode + cen]
            ax.plot(range(nL), hon, "o" + ls, ms=3, c="tab:blue", label=f"honest answer judged TRUE ({mode})")
            ax.plot(range(nL), lie, "o" + ls, ms=3, c="tab:red", label=f"requested lie judged FALSE ({mode})")
        ax.axhline(0.5, c="gray", ls="--", lw=1); ax.axvline(Lb, c="gray", ls=":", lw=1)
        ax.set_ylim(-0.02, 1.02); ax.set_xlabel("layer"); ax.set_ylabel("rate"); ax.legend(fontsize=8)
        ax.set_title(("raw" if not cen else "centered (mean of all generated answers shifted to training mean)"), fontsize=10)
    fig.suptitle(f"Lie detector on the model's own answers  {tag}  "
                 f"(honest n={keep_h.sum()}, lie n={keep_l.sum()}, probe: city-prefecture, per layer)", fontsize=11)
    fig.tight_layout(); fig.savefig(f"{args.out}/extra_a_{tag}.png", dpi=150); plt.close(fig)
    print(f"保存：{args.out}/extra_a_{tag}.json/png, extra_answers_{tag}.csv")

# ---------- b：知らない話（架空の市）では探知機が働かない？ ----------
if "b" in EXTRA:
    Lx = args.extra_layer if args.extra_layer is not None else int(lay.acc_city_pref.idxmax())
    # 架空の市名：地名によく使う漢字を3つ組み合わせ、GeoNames の日本の地名（全言語の別名）に無いものだけ
    allnames = set(pd.read_csv("JP_alt.txt", sep="\t", header=None, dtype=str, keep_default_na=False,
                               quoting=3)[3]) | set(df.name) | set(df.ja)
    KAN = list("山川田野原沢井谷島崎岡浜松杉森林木本村北南東西中上下大小高長石橋平新富福吉岩倉宮瀬根津江坂"
               "鳴戸菅柳桜梅竹藤朝日月星青白赤黒金銀泉湖淵滝峰岳嶺里郷城宿堀塚")
    r2 = np.random.default_rng(args.seed + 1)
    fake = []
    while len(fake) < 50:
        nm = "".join(r2.choice(KAN, 3, replace=False))
        if nm in allnames or nm + "市" in allnames or nm + "町" in allnames or nm + "村" in allnames: continue
        if any(nm in c for c in cities.ja) or nm + "市" in fake: continue
        fake.append(nm + "市")
    fdf = pd.DataFrame({"city": fake, "pref": r2.choice(PREF, len(fake))})
    fdf["text"] = [f"{c}は{p}にある。" for c, p in zip(fdf.city, fdf.pref)]
    XF = last_token_states(fdf.text)[:, Lx]
    fdf["p_true"] = p_true(XF, Lx)
    fdf.to_csv(f"{args.out}/extra_b_fake_cities_{tag}.csv", index=False)
    # 実在の文：答えを見ていない線で当てる（5分割、層 Lx）
    pr = np.zeros(len(yc))
    for tr, te in skf.split(Xc[:, Lx], yc):
        sc, m = fit(Xc[tr, Lx], yc[tr])
        pr[te] = m.predict_proba(sc.transform(Xc[te, Lx]))[:, 1]
    conf_t, conf_f, conf_x = np.abs(pr[yc == 1] - .5), np.abs(pr[yc == 0] - .5), np.abs(fdf.p_true.to_numpy() - .5)
    ok = (pr > 0.5) == yc
    kn = ans.knows.to_numpy()  # 本番の都市と県の i 番目の文は cities の i 番目の都市
    res_b = {"model": args.model, "layer": Lx, "n_fake": len(fdf), "fake_examples": fake[:10],
             "mean_conf_real_true": round(float(conf_t.mean()), 3), "mean_conf_real_false": round(float(conf_f.mean()), 3),
             "mean_conf_fake": round(float(conf_x.mean()), 3),
             "fake_judged_true_rate": round(float((fdf.p_true > 0.5).mean()), 3),
             "model_knows_rate": round(float(kn.mean()), 3), "n_known": int(kn.sum()), "n_unknown": int((~kn).sum()),
             "detector_acc_known": round(float(ok[kn].mean()), 3) if kn.any() else None,
             "detector_acc_unknown": round(float(ok[~kn].mean()), 3) if (~kn).any() else None}
    print(json.dumps(res_b, ensure_ascii=False, indent=1))
    json.dump(res_b, open(f"{args.out}/extra_b_{tag}.json", "w"), ensure_ascii=False, indent=1)
    fig, axs = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [2, 1]})
    bins = np.linspace(0, 0.5, 21)
    for v, c, lb in [(conf_t, "tab:blue", "real, true"), (conf_f, "tab:red", "real, false"), (conf_x, "gray", "fictional city")]:
        axs[0].hist(v, bins=bins, density=True, alpha=0.45, color=c, label=f"{lb} (n={len(v)}, mean={v.mean():.2f})")
    axs[0].set_xlabel("confidence |p - 0.5|"); axs[0].set_ylabel("density"); axs[0].legend()
    axs[0].set_title(f"How sure is the detector? (layer {Lx})")
    va = [res_b["detector_acc_known"] or 0, res_b["detector_acc_unknown"] or 0]
    axs[1].bar(["model knows\nthe city", "model does not\nknow the city"], va, color=["purple", "silver"])
    for k, (v, n) in enumerate(zip(va, [res_b["n_known"], res_b["n_unknown"]])):
        axs[1].text(k, v + 0.02, f"{v:.2f}\n(n={n})", ha="center")
    axs[1].axhline(0.5, c="gray", ls="--", lw=1); axs[1].set_ylim(0, 1.1); axs[1].set_ylabel("detector accuracy (5-fold CV)")
    axs[1].set_title(f"knows vs not (layer {Lx})")
    fig.suptitle(tag); fig.tight_layout(); fig.savefig(f"{args.out}/extra_b_{tag}.png", dpi=150); plt.close(fig)
    print(f"保存：{args.out}/extra_b_{tag}.json/png, extra_b_fake_cities_{tag}.csv")


## 1. 実行（本番＋点の保存）

In [ ]:
!python truth_probe.py --out results --save_proj

## 2. 確かめる（層9までは青と赤が混ざり、層10・20 で左右に分かれていればOK。紫の線の右が本当と判定）


In [ ]:
import numpy as np, matplotlib.pyplot as plt
d = np.load('results/layers_proj_Qwen2.5-7B-Instruct.npz')
print({k: d[k].shape for k in d})
xy, lab = d['city_pref_xy'], d['city_pref_label']
fig, axs = plt.subplots(1, 4, figsize=(16, 4))
for ax, L in zip(axs, [0, 9, 10, 20]):
    ax.scatter(xy[L, lab == 1, 0], xy[L, lab == 1, 1], s=6, c='tab:blue', label='true')
    ax.scatter(xy[L, lab == 0, 0], xy[L, lab == 0, 1], s=6, c='tab:red', label='false')
    ax.axvline(0, c='purple'); ax.set_title(f'layer {L}')
axs[0].legend(); plt.show()
for L in [0, 9, 10, 20]:
    print(L, 'あたり', round(float(((xy[L, :, 0] > 0) == (lab == 1)).mean()), 3))

## 3. ダウンロード（この zip を results_colab/ に置いてください）

In [ ]:
!zip -q -j layers_proj.zip results/layers_proj_*.npz results/layers_*.csv
from google.colab import files; files.download('layers_proj.zip')